# Juno MAG and SPICE download

Downloads Juno data into folders parallel to the Cassini data, using `jupiter/juno_download.py`.

| Step | What | Destination | Size (checked 2026-11-22) |
|---|---|---|---|
| 0 | Archive docs and file index | `data/raw/PDS/juno/docs/` | 9,984,318 B (about 10 MB) |
| 1 | Moon flybys, full resolution (Ganymede, Europa, Io ×2) | `data/raw/PDS/juno/mag_pc_flyby/` | 170,433,650 B (0.16 GiB) |
| 2 | SPICE kernels for positions | `data/raw/spice_kernels/spice-juno/` | 277,116,730 B (0.26 GiB) |
| 4 | 1 s MAG, whole mission (3,488 days) | `data/raw/PDS/juno/mag_pc_1s/` | 38,203,181,847 B (35.58 GiB) |
| 5 | Native MAG, selected days | `data/raw/PDS/juno/mag_pc_native/` | about 0.1–0.7 GiB per day |
| 6 | Native MAG, whole mission (July 2016 onward) | `data/raw/PDS/juno/mag_pc_native/` | about 382 GiB |

**Dry run** (`DRY_RUN = True`): asks the server for each file's name and size, prints the count and total, and writes nothing.
**Real run** (`DRY_RUN = False`): downloads each file with its `.lbl` label. Complete files are skipped, so an interrupted run can be repeated.

**Updates:** every step is incremental. A file on disk is kept when the server listing shows the same name, a matching size and no newer date. Rerunning months later downloads only new orbits, newly added moon flybys, reprocessed files, and SPICE kernels NAIF has extended. Use the **Update** cell at the end for this.

In [1]:
# Make the project folder importable whether this notebook runs from saturn2025/ or saturn2025/jupiter/.
import sys
from pathlib import Path

PROJECT = Path.cwd() if (Path.cwd() / "jupiter").is_dir() else Path.cwd().parent
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

from jupiter import juno_download as jd

print("Project:", PROJECT)
print("Juno MAG folder:", jd.JUNO_PDS_DIR)
print("Juno SPICE folder:", jd.JUNO_SPICE_DIR)

Project: /Users/chengzhimao/Documents/Code/saturn2025
Juno MAG folder: /Users/chengzhimao/Documents/Code/saturn2025/data/raw/PDS/juno
Juno SPICE folder: /Users/chengzhimao/Documents/Code/saturn2025/data/raw/spice_kernels/spice-juno


In [2]:
# True: list files and sizes only. False: download.
DRY_RUN = True

## Step 0: docs and file index

In [3]:
jd.download_juno_docs(dry_run=DRY_RUN)

Step 0 docs: checking:   0%|          | 0/22 [00:00<?, ?it/s]

Step 0 docs: 22 files, 0.009 GiB in total
  on disk and current: 22 files, 0.009 GiB
  new or changed:      0 files, 0 bytes = 0.000 GiB
  dry run, nothing written. Pass dry_run=False to download.


{'files': 22, 'current': 22, 'new': 0, 'bytes_new': 0, 'downloaded': 0}

## Step 1: moon flybys (full resolution, about 1 h around each closest approach)

Finds every moon-flyby product in the archive, including flybys added after this notebook was written.

In [4]:
print(list(jd.FLYBYS))
jd.download_juno_flybys(dry_run=DRY_RUN)

['ganymede_2021-06-07', 'europa_2022-09-29', 'io_2023-12-30', 'io_2024-02-03']


KeyboardInterrupt: 

## Step 2: SPICE kernels

In [ ]:
jd.download_juno_spice(dry_run=DRY_RUN)

Step 2 SPICE: sizing:   0%|          | 0/7 [00:00<?, ?it/s]

Step 2 SPICE: 7 files, 277,116,730 bytes = 0.258 GiB


Step 2 SPICE:   0%|          | 0.00/277M [00:00<?, ?B/s]

  7 downloaded, 0 already complete


{'files': 7, 'bytes': 277116730, 'downloaded': 7, 'skipped': 0}

## Step 4: 1 s MAG, whole mission (35.6 GiB, several hours)

The first run checks the size of each of about 7,000 files on the server (about 30 s). Later runs skip files the listing shows as unchanged.

In [ ]:
jd.download_juno_mag("r1s", dry_run=DRY_RUN)

MAG r1s: sizing:   0%|          | 0/6976 [00:00<?, ?it/s]

MAG r1s: 6976 files, 38,203,181,847 bytes = 35.579 GiB


MAG r1s:   0%|          | 0.00/38.2G [00:00<?, ?B/s]

Exception in thread Thread-4:
Traceback (most recent call last):
  File "/opt/anaconda3/envs/saturn2025/lib/python3.13/threading.py", line 1043, in _bootstrap_inner
    self.run()
    ~~~~~~~~^^
  File "/opt/anaconda3/envs/saturn2025/lib/python3.13/site-packages/tqdm/_monitor.py", line 84, in run
    instance.refresh(nolock=True)
    ~~~~~~~~~~~~~~~~^^^^^^^^^^^^^
  File "/opt/anaconda3/envs/saturn2025/lib/python3.13/site-packages/tqdm/std.py", line 1347, in refresh
    self.display()
    ~~~~~~~~~~~~^^
  File "/opt/anaconda3/envs/saturn2025/lib/python3.13/site-packages/tqdm/notebook.py", line 171, in display
    rtext.value = right
    ^^^^^^^^^^^
  File "/opt/anaconda3/envs/saturn2025/lib/python3.13/site-packages/traitlets/traitlets.py", line 716, in __set__
    self.set(obj, value)
    ~~~~~~~~^^^^^^^^^^^^
  File "/opt/anaconda3/envs/saturn2025/lib/python3.13/site-packages/traitlets/traitlets.py", line 706, in set
    obj._notify_trait(self.name, old_value, new_value)
    ~~~~~~~~~~~

  6976 downloaded, 0 already complete


{'files': 6976, 'bytes': 38203181847, 'downloaded': 6976, 'skipped': 0}

## Step 5 (optional): native MAG for selected days

Example: the two days around the 2023-12-30 Io flyby. Day codes are `YYYYDDD`.
Downloading every native day (about 382 GiB) is blocked unless you name orbits or days.

In [ ]:
jd.download_juno_mag("native", perijoves=[57], days=["2023364", "2023365"], dry_run=DRY_RUN)

## Step 6: native (full-resolution) MAG, whole mission (about 382 GiB)

`YEARS = "all"` downloads every native daily file since orbit insertion (July 2016 onward), including years added in later PDS releases.
To download in parts, use one year (`YEARS = 2017`) or a list (`YEARS = [2016, 2017]`).
`WORKERS` is the number of files downloaded at the same time (4 by default).
This cell has its own switch so that setting `DRY_RUN = False` above does not start a 382 GiB download.
The dry run prints what is already on disk and what is still to download, by year, and writes nothing.
The download refuses to start if less than 20 GiB would stay free. If it stops, run the cell again: it continues with the missing files.

In [ ]:
DRY_RUN_NATIVE_ALL = False   # set to False only when you are ready for ~382 GiB
YEARS = 'all'               # or 2017, or [2016, 2017]
WORKERS = 4                 # files downloaded at the same time
jd.download_juno_native_all(years=YEARS, dry_run=DRY_RUN_NATIVE_ALL, workers=WORKERS)

MAG native: checking:   0%|          | 0/6976 [00:00<?, ?it/s]

MAG native: 6976 files, 381.626 GiB in total
  on disk and current: 0 files, 0.000 GiB
  new or changed:      6976 files, 409,767,869,898 bytes = 381.626 GiB
  to download, by year:
    2016:    34.41 GiB
    2017:    42.53 GiB
    2018:    35.83 GiB
    2019:    36.92 GiB
    2020:    35.66 GiB
    2021:    36.29 GiB
    2022:    35.32 GiB
    2023:    38.65 GiB
    2024:    38.02 GiB
    2025:    36.02 GiB
    2026:    11.96 GiB


MAG native:   0%|          | 0.00/410G [00:00<?, ?B/s]

ChunkedEncodingError: ('Connection broken: IncompleteRead(46317149 bytes read, 42856116 more expected)', IncompleteRead(46317149 bytes read, 42856116 more expected))

## Update: fetch only what is new (run every few months)

Checks docs, moon flybys, SPICE, 1 s MAG and native MAG against the server and lists only new or changed files.
Set `native=False` to skip the full-resolution data. `verify=True` asks the server for the exact size of every file (slower, for a full integrity check).
Older versions replaced on the server (v01 → v02) are listed but never deleted.

In [ ]:
DRY_RUN_UPDATE = True   # False downloads the new files
jd.update_juno(dry_run=DRY_RUN_UPDATE, native=True)

## Check what is on disk

In [ ]:
def folder_size(path):
    files = [f for f in Path(path).rglob("*") if f.is_file() and not f.name.endswith(".part")]
    total = sum(f.stat().st_size for f in files)
    return len(files), total

for folder in [jd.DOCS_DIR, jd.FLYBY_DIR, jd.JUNO_SPICE_DIR, *jd.MAG_DIRS.values()]:
    if folder.exists():
        n, total = folder_size(folder)
        print(f"{folder.relative_to(PROJECT)}: {n} files, {total:,} bytes = {total / 2**30:.3f} GiB")
    else:
        print(f"{folder.relative_to(PROJECT)}: not created yet")

data/raw/PDS/juno/docs: 22 files, 9,984,318 bytes = 0.009 GiB
data/raw/PDS/juno/mag_pc_flyby: 8 files, 170,433,650 bytes = 0.159 GiB
data/raw/spice_kernels/spice-juno: 7 files, 277,116,730 bytes = 0.258 GiB
data/raw/PDS/juno/mag_pc_1s: 6976 files, 38,203,181,847 bytes = 35.579 GiB
data/raw/PDS/juno/mag_pc_60s: not created yet
data/raw/PDS/juno/mag_pc_native: not created yet
